# L5a: Multiple Asset Geometric Brownian Motion
In this lecture, we extend the single asset geometric Brownian motion (SAGBM) model of L4b to many correlated assets. Real firms do not move independently: sectors share inputs, everything responds to market news, and hedges move against each other. Capturing that co-movement takes one new object, the covariance matrix, and one new device, a matrix square root that turns independent shocks into correlated ones. We then estimate the covariance from data and, looking ahead to portfolios, introduce a way to sample portfolio weights.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * **Model correlated assets with multiple asset GBM:** Write the multiple asset GBM stochastic differential equation with a shared shock vector and a covariance factor (a loading matrix whose product with its transpose is the covariance rate), state the exact one-step transition, and explain why the factorization reproduces the covariance.
> * **Estimate the covariance matrix from data:** Form the growth-rate data matrix, center it, compute the sample covariance and correlation, convert the growth-rate covariance to the GBM covariance rate with the time step, and state the structural properties every covariance estimate must satisfy.
> * **Describe and sample portfolio weights:** Define buy-and-hold portfolio wealth for a weight vector on the long-only simplex, define the Dirichlet distribution and its moments, and explain why sampling weights explores the simplex but does not optimize anything.

Let's get started!
___


## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Implement an out-of-sample (OOS) single asset prediction](CHEME-5660-L5a-Example-OOS-SAGBM-Fall-2026.ipynb). Use parameters estimated from 2014 to 2024 data with a single asset GBM model to predict prices the model never saw (calendar year 2025), and check how often the observed price stays inside the model's prediction bands.

The second example builds the central object of the lecture:

> [▶ Compute the covariance matrix for our dataset](CHEME-5660-L5a-Example-CovarianceMatrix-Fall-2026.ipynb). Compute the empirical growth-rate covariance matrix for the firms in our dataset, convert it to the GBM covariance rate, verify it against built-in functions and the volatilities we estimated in L4b, and examine the covariance and correlation of a pair of firms.

The third example looks ahead to portfolios:

> [▶ Sample portfolio weights with the Dirichlet distribution](CHEME-5660-L5a-Example-Dirichlet-PortfolioWeights-Fall-2026.ipynb). Draw random long-only weight vectors for a small portfolio, see how the concentration parameter shapes them, and compute the growth-rate mean and variance of the resulting portfolios.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___


## Concept Review: Single Asset GBM Model
Single asset geometric Brownian motion (GBM) is a stochastic differential equation describing the share price $S(t)$ as a continuous-time random walk whose drift and noise are both proportional to $S(t)$:
$$
\begin{align*}
\frac{dS\left(t\right)}{S(t)} &= \mu\,{dt}+\sigma\,{dW(t)}\\
\end{align*}
$$
Here, $\mu\in\mathbb{R}$ (units: inverse years) is the arithmetic GBM drift, $\sigma>0$ (units: inverse years to the one-half power) is the constant __volatility__, and $dW(t)$ is the increment of a Wiener process. 

> __Parameters__
> 
> * __Drift versus growth:__ The parameter $\mu$ is the arithmetic drift in the price SDE. The __mean growth rate__ $\bar g=\mu-\sigma^{2}/2$ (units: inverse years) is the expected value of the one-step growth rate $g_{j}=(1/\Delta{t})\ln(S_{t_{j}}/S_{t_{j-1}})$; equivalently $\mu=\bar g+\sigma^2/2$. The two share units but are different quantities.
> * __Risk:__ The volatility $\sigma$ sets the standard deviation of the one-step growth rate, $\sigma/\sqrt{\Delta{t}}$, and of the one-step log return, $\sigma\sqrt{\Delta{t}}$. We estimate it from the growth-rate standard deviation as $\hat{\sigma} = \sigma_{g}\sqrt{\Delta{t}}$.
> * __Constancy:__ Both parameters are assumed constant over time, although in practice they vary; the out-of-sample example below shows what that assumption costs.

On a grid $t_{j} = j\Delta{t}$, the price advances one step at a time by the exact __one-step transition__ from L4b:
$$
\boxed{
\begin{align*}
S_{t_{j}} &= S_{t_{j-1}}\;\exp\Biggl[\bar g\,\Delta{t} + \sigma\sqrt{\Delta{t}}\;Z_{j}\Biggr]\qquad{j=1,2,\dots,N}\\
\end{align*}}
$$
where the shocks $Z_{1},Z_{2},\dots$ are independent standard normal random variables. For a single fixed horizon $T$ the same solution reads $S_{T}=S_{0}\exp[\bar g\,T+\sigma\sqrt{T}\,Z]$ with one standard normal $Z$; the one-step form is what we use to simulate paths. In L4b we estimated $\bar g$ and $\sigma$ from data the model had already seen. Let's look at what happens when we ask the model about prices it has __not__ seen.

> __Example__
>
> [▶ Implement an out-of-sample (OOS) single asset prediction](CHEME-5660-L5a-Example-OOS-SAGBM-Fall-2026.ipynb). Use parameters estimated from 2014 to 2024 data with a single asset GBM model to predict prices the model never saw (calendar year 2025), and check how often the observed price stays inside the model's prediction bands.

### Catch-up: the GBM trade rule
If we did not reach the trade-rule probability at the end of L4b, we pick it up now. For a scheduled long position in $n_{0}$ shares bought at $S_{0}$ and sold at $S_{T}$ after a holding period $T>0$, with benchmark growth rate $g_{b}$, the scaled NPV is $\rho_{T}=\texttt{NPV}(g_{b},T)/(n_{0}S_{0})=(S_{T}/S_{0})e^{-g_{b}T}-1$. For a target $\rho_{\star}>-1$, the single asset GBM model gives the closed-form probability that the trade clears the target:
$$
\boxed{
\mathbb{P}\left(\rho_{T}>\rho_{\star}\right) = 1-\Phi\!\left(\frac{\ln(1+\rho_{\star}) + g_{b}T - \bar g\,T}{\sigma\sqrt{T}}\right)
}
$$
where $\Phi(\cdot)$ is the standard normal cumulative distribution function. The derivation is in the [GBM Trade Rule section of the L4b lecture](../../week-4/L4b/CHEME-5660-L4b-Lecture-SingleAsset-GeometricBrownianMotion-TradeRule-Fall-2026.ipynb), and the [L4b trade-rule example](../../week-4/L4b/CHEME-5660-L4b-Example-GBM-NPV-TradeRule-Fall-2026.ipynb) evaluates it for a firm in our dataset. Everything in today's lecture is about extending the price model itself, from one asset to many.
___


## Multiple Asset Geometric Brownian Motion (MAGBM) Model
We could simulate $M$ single asset GBM models side by side, but independently simulated assets never move together, and co-movement is the whole point of a portfolio: firms respond together to market news, sectors share inputs, and hedges move against each other. Consider a portfolio $\mathcal{P}=\{1,2,\dots,M\}$ of $M$ assets. From here on, $i$ and $j$ index assets, $\ell$ indexes the coordinates of the shock vector, and (in the data section) $k$ indexes time periods; the single asset review above kept L4b's $t_{j}$ and $Z_{j}$. Each asset $i\in\mathcal{P}$ has an arithmetic GBM drift $\mu_{i}$ (units: inverse years), and the assets share a __covariance rate__ $\mathbf{C}\in\mathbb{R}^{M\times M}$ (units: inverse years) whose diagonal entries are the squared volatilities $C_{ii}=\sigma_{i}^{2}$ and whose off-diagonal entries measure how the log returns of two assets move together per unit time. To generate correlated shocks from independent ones, we introduce a __loading matrix__ $\mathbf{A}\in\mathbb{R}^{M\times M}$ (units: inverse years to the one-half power) with $\mathbf{A}\mathbf{A}^{\top} = \mathbf{C}$, and $M$ independent standard Wiener processes $W_{1}(t),\dots,W_{M}(t)$ (so $dW_{\ell}\,dW_{\ell^{\prime}}=\delta_{\ell\ell^{\prime}}\,dt$). The multiple asset GBM model for the share price $S_{i}(t)$ of asset $i$ is then:
$$
\begin{equation*}
\frac{dS_{i}\left(t\right)}{S_{i}(t)} = \mu_i\,{dt}+\sum_{\ell=1}^{M}A_{i\ell}\;{dW_{\ell}(t)}\qquad\text{for}\quad{i\in\mathcal{P}}
\end{equation*}
$$
Applying Itô's lemma to $\ln S_{i}(t)$ gives each asset its own half-variance correction, so the mean growth rate of asset $i$ is $\bar g_i=\mu_i-C_{ii}/2$, and $\mu_i=\bar g_i+C_{ii}/2$, exactly as in the single asset case with $\sigma_{i}^{2}=C_{ii}$.

> __Why do we need the $\mathbf{A}\mathbf{A}^{\top}$ factorization?__ 
>
> A factorization $\mathbf{A}\mathbf{A}^{\top} = \mathbf{C}$ is called a __Cholesky decomposition__ when $\mathbf{A}$ is lower triangular; the symmetric positive semidefinite square root $\mathbf{C}^{1/2}$ is another valid choice. Any __covariance factor__ $\mathbf{A}$ with $\mathbf{A}\mathbf{A}^{\top}=\mathbf{C}$ will do (a matrix $\mathbf{B}$ with $\mathbf{B}^{2}=\mathbf{C}$ is not enough unless it is also symmetric). We need it because:
> 
> * __Independent noise becomes correlated noise__: The Wiener processes $W_{\ell}(t)$ are independent, but real assets move together. Multiplying the vector of independent increments by $\mathbf{A}$ mixes them into the correlated structure we observe.
> 
> * __It reproduces the covariance rate__: Over an interval $dt$, the covariance of the noise terms of assets $i$ and $j$ is:
> $$\text{Cov}\left(\sum_{\ell}A_{i\ell}\,dW_{\ell}, \sum_{\ell}A_{j\ell}\,dW_{\ell}\right) = \sum_{\ell}A_{i\ell}A_{j\ell}\,dt = (\mathbf{A}\mathbf{A}^{\top})_{ij}\,dt = C_{ij}\,dt$$
> so $\mathbf{C}$ is the covariance per unit time (the increment covariance is $C_{ij}\,dt$), and any covariance factor $\mathbf{A}$ of $\mathbf{C}$ produces it. __TL;DR:__ the factorization is what makes the simulated assets have the covariance we asked for.

As in L4b, the model can be advanced one step at a time exactly. Let $\mathbf{Z}\sim\mathcal{N}(\mathbf{0},\mathbf{I}_{M})$ be a vector of $M$ independent standard normal shocks, drawn __fresh at every step__ and __shared by all assets__ within that step. The exact one-step transition for asset $i$ over a step $\Delta{t}$ is:
$$
\boxed{
\begin{align*}
S_{i}(t+\Delta{t}) &= S_{i}(t)\cdot\exp\Biggl[\bar g_{i}\,\Delta{t} + \sqrt{\Delta{t}}\cdot\left(\mathbf{A}\mathbf{Z}\right)_{i}\Biggr]\qquad{i\in\mathcal{P}}\\
\end{align*}}
$$
where $\left(\mathbf{A}\mathbf{Z}\right)_{i}=\sum_{\ell=1}^{M}A_{i\ell}Z_{\ell}$ is the correlated shock of asset $i$. Two things carry over from L4b. First, this transition is exact at the grid points for constant parameters; it is not a time-stepping approximation. Second, the shocks must be redrawn at every step: a path is built from independent increments. (A single fixed horizon $T$ can instead be sampled directly, with one shock vector scaled by $\sqrt{T}$, exactly as in the single asset case.)

Dividing the transition by $S_{i}(t)$ and taking logarithms gives the one-step log return $r_{i}=\ln(S_{i}(t+\Delta{t})/S_{i}(t))$ of each asset over the same step; dividing by $\Delta{t}$ gives the one-step growth rate $g_{i}=r_{i}/\Delta{t}$. Collecting them in vectors $\mathbf{r}$ and $\mathbf{g}=\mathbf{r}/\Delta{t}$ gives the __scaling rule__ that connects the model to data:
$$
\begin{align*}
\mathbf{g} &= \bar{\mathbf{g}} + \frac{1}{\sqrt{\Delta{t}}}\,\mathbf{A}\mathbf{Z}
\quad\Longrightarrow\quad
\mathbb{E}\left[\mathbf{g}\right] = \bar{\mathbf{g}} = \boldsymbol{\mu}-\tfrac{1}{2}\operatorname{diag}(\mathbf{C}),
\qquad
\underbrace{\text{Cov}\left(\mathbf{g}\right)}_{\text{growth-rate covariance}} = \frac{\mathbf{C}}{\Delta{t}},
\qquad
\underbrace{\text{Cov}\left(\mathbf{r}\right)}_{\text{log-return covariance}} = \mathbf{C}\,\Delta{t}
\end{align*}
$$
The growth-rate covariance (units: inverse years squared), the log-return covariance (dimensionless), and the covariance rate $\mathbf{C}$ (inverse years) are three different matrices related by powers of $\Delta{t}$; keeping them apart is the multi-asset version of L4b's $\hat{\sigma}=\sigma_{g}\sqrt{\Delta{t}}$. However, before we can use the MAGBM model, we need $\mathbf{C}$, and $\mathbf{C}$ comes from data. Let's dig into the key new concept: the covariance matrix.
___


## Empirical Covariance Matrix
The covariance matrix is a key concept in statistics and machine learning that describes the relationships between the features in a dataset. In our case, the features are the firms in our portfolio $\mathcal{P}$, and the samples are their historical growth rates over time (e.g., daily). Positive covariance means two firms tend to be above (or below) their means together, negative covariance means one tends to be above its mean when the other is below, and zero covariance means no linear relationship. The figure shows samples from three two-dimensional distributions with negative, zero, and positive covariance; the lower row repeats each cloud with the covariance multiplied by four, which doubles the spread but leaves the direction of co-movement unchanged.

<div>
    <center>
        <img src="figs/Fig-Cov-Schematic.png" width="880" alt="Six scatter plots of two-dimensional samples: the top row shows clouds with negative, zero, and positive covariance, tilted down, round, and tilted up respectively; the bottom row repeats each cloud with four times the covariance, so the points spread twice as far while keeping the same orientation"/>
    </center>
</div>

Suppose we have $N\geq2$ equally spaced time periods of growth-rate data (e.g., $N$ trading days) for the $M$ firms in $\mathcal{P}$, aligned so that every firm's growth rate in period $k$ covers the same interval. Let $g_k^{(i)}$ be the growth rate of firm $i$ in time period $k=1,2,\dots,N$; we index firms by $i$ and $j$ and time periods by $k$, as L5b does. Collect the time series for firm $i$ into the vector $\mathbf{g}^{(i)}=[g_1^{(i)},\dots,g_N^{(i)}]^\top$, and let $g^{\prime}_{i}=\frac{1}{N}\sum_{k=1}^{N}g_{k}^{(i)}$ be its sample mean (the sample mean $g^{\prime}$ of L3a and L4b, one per firm). The empirical growth-rate covariance matrix $\hat{\mathbf{\Sigma}}_{g}\in\mathbb{R}^{M\times M}$ is the square symmetric matrix of pairwise covariances:
$$
\begin{align*}
    \hat{\Sigma}_{g,ij} &= \frac{1}{N-1}\sum_{k=1}^{N}\overbrace{\bigl(g^{(i)}_k-g^{\prime}_{i}\bigr)}^{\text{deviation from mean}}\,\bigl(g^{(j)}_k-g^{\prime}_{j}\bigr)\qquad{i,j\in\mathcal{P}}\\
\end{align*}
$$
The diagonal entry $\hat{\Sigma}_{g,ii}$ is the sample variance of firm $i$'s growth rate, so $\sqrt{\hat{\Sigma}_{g,ii}}$ is the growth-rate standard deviation $\sigma_{g}$ of L3a for firm $i$ (units: inverse years), not the volatility. The sign of an off-diagonal entry gives the direction of the relationship, but its magnitude depends on the scale of both firms' growth rates. The standardized strength is the __correlation__:
$$
\boxed{
\begin{align*}
\rho_{ij} &= \frac{\hat{\Sigma}_{g,ij}}{\sqrt{\hat{\Sigma}_{g,ii}\,\hat{\Sigma}_{g,jj}}}\in\left[-1,1\right]
\quad\Longleftrightarrow\quad
\hat{\Sigma}_{g,ij} = \rho_{ij}\sqrt{\hat{\Sigma}_{g,ii}\,\hat{\Sigma}_{g,jj}}\quad\blacksquare
\end{align*}}
$$
defined whenever both variances are positive. Correlation is what to quote when comparing the strength of relationships across pairs; covariance is what the model needs. Zero correlation means no __linear__ relationship in this sample; it does not imply the two firms are independent.

### The data matrix and the sample covariance
We do not compute the covariance one pair at a time. Arrange the growth rates in a __data matrix__ $\mathbf{G}\in\mathbb{R}^{N\times M}$ (rows are time periods, columns are firms), where row $k$ holds the growth rates of all $M$ firms in period $k$:
$$
\mathbf{G} = \begin{bmatrix}
g_1^{(1)} & g_1^{(2)} & \cdots & g_1^{(M)} \\
g_2^{(1)} & g_2^{(2)} & \cdots & g_2^{(M)} \\
\vdots & \vdots & \ddots & \vdots \\
g_N^{(1)} & g_N^{(2)} & \cdots & g_N^{(M)}
\end{bmatrix}
$$
To center the data, subtract each firm's sample mean from its column. Let $\mathbf{g}^{\prime} = [g^{\prime}_{1}, g^{\prime}_{2}, \ldots, g^{\prime}_{M}]^{\top}$ be the vector of sample means. The centered data matrix is:
$$
\tilde{\mathbf{G}} = \mathbf{G} - \mathbf{1}\,\mathbf{g}^{\prime\top}
$$
where $\mathbf{1} \in \mathbb{R}^{N}$ is a vector of ones, so that $\mathbf{1}\,\mathbf{g}^{\prime\top}$ is an $N \times M$ matrix whose every row is the vector of sample means. 

> __Outer product:__ The matrix $\mathbf{1}\,\mathbf{g}^{\prime\top}$ is an example of an outer product. The [outer product](https://en.wikipedia.org/wiki/Outer_product) of two vectors $\mathbf{a} \in \mathbb{R}^{N}$ and $\mathbf{b} \in \mathbb{R}^{M}$ is the $N \times M$ matrix $\mathbf{a}\mathbf{b}^{\top}$ with elements $(\mathbf{a}\mathbf{b}^{\top})_{kj} = a_k b_j$. 

The empirical growth-rate covariance matrix is then one matrix product:
$$
\boxed{
\hat{\mathbf{\Sigma}}_{g} = \frac{1}{N-1}\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}
}
$$

### From the growth-rate covariance to the GBM covariance rate
The scaling rule from the MAGBM section says $\text{Cov}(\mathbf{g})=\mathbf{C}/\Delta{t}$. Therefore the estimate of the GBM covariance rate, the matrix the model needs, is:
$$
\boxed{
\hat{\mathbf{C}} = \Delta{t}\;\hat{\mathbf{\Sigma}}_{g}
}
$$
Its diagonal gives the volatility estimates $\hat{\sigma}_{i}=\sqrt{\hat{C}_{ii}}=\sqrt{\Delta{t}}\sqrt{\hat{\Sigma}_{g,ii}}$, which is exactly L4b's $\hat{\sigma}=\sigma_{g}\sqrt{\Delta{t}}$ firm by firm, and the log-return covariance is $\hat{\mathbf{\Sigma}}_{r}=\Delta{t}^{2}\,\hat{\mathbf{\Sigma}}_{g}=\Delta{t}\,\hat{\mathbf{C}}$. For daily data, $\Delta{t}=1/252$ years: multiplying a daily log-return covariance by 252 estimates $\hat{\mathbf{C}}$; it does not estimate $\hat{\mathbf{\Sigma}}_{g}$. Correlations are the same for all three matrices, because the scaling cancels in the ratio.

> __Covariance Matrix Properties:__
>
> Every covariance estimate must satisfy the following, and it is worth checking them numerically:
> * __Elements__: The diagonal elements $\hat{\Sigma}_{g,ii}$ are the growth-rate variances of the firms (non-negative); the off-diagonal elements $\hat{\Sigma}_{g,ij}$, $i\neq{j}$, are the covariances between firms, whose sign gives the direction of the linear relationship. The same holds for $\hat{\mathbf{C}}$.
> * __Symmetry__: $\hat{\Sigma}_{g,ij} = \hat{\Sigma}_{g,ji}$ for all $i$ and $j$, directly from the definition; $\hat{\mathbf{C}}=\Delta{t}\,\hat{\mathbf{\Sigma}}_{g}$ inherits symmetry and positive semidefiniteness.
> * __Positive semidefinite__: For any weight vector $\mathbf{v} \in \mathbb{R}^{M}$, we have $\mathbf{v}^{\top}\hat{\mathbf{\Sigma}}_{g}\mathbf{v} = \frac{1}{N-1}\lVert\tilde{\mathbf{G}}\mathbf{v}\rVert_{2}^{2}\geq 0$, because $\mathbf{v}^{\top}\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}\mathbf{v}$ is a squared length. This is why a portfolio variance $\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g}\mathbf{w}$ can never come out negative, and it is the property the optimization in L5b rests on.

The estimate can also fail in ways that the properties above do not rule out, and a graduate treatment should say so before calling $\hat{\mathbf{C}}$ simulation-ready.

> __Caveats:__
>
> * __Only semidefinite:__ Duplicated assets, an exact linear dependence, or too few observations make $\hat{\mathbf{\Sigma}}_{g}$ singular; because the centered matrix $\tilde{\mathbf{G}}$ has rank at most $N-1$, the estimate is always singular once $M\geq N$. An unpivoted Cholesky factorization then fails; an eigenvalue or singular-value square root still works. Adding a small diagonal "jitter" restores invertibility but changes the model, and must be documented, not hidden.
> * __Sampling error:__ A matrix with $M(M+1)/2$ distinct entries estimated from $N$ observations is noisy when $M$ is not small relative to $N$; pairwise estimates shift across windows, and correlations change in stressed markets. Shrinkage and factor models trade flexibility for stability; the optional advanced material takes this up.

Let's compute the empirical covariance matrix for the firms in our dataset.

> __Example__
> 
> [▶ Compute the covariance matrix for our dataset](CHEME-5660-L5a-Example-CovarianceMatrix-Fall-2026.ipynb). Compute the empirical growth-rate covariance matrix for the firms in our dataset, convert it to the GBM covariance rate, verify it against built-in functions and the volatilities we estimated in L4b, and examine the covariance and correlation of a pair of firms.

___


## Portfolio Weights and Dirichlet Sampling
Once we can simulate several assets together, the natural next question is how much of our money to put in each. Let total initial wealth be $W_{0}>0$ dollars, and let $\mathbf{w}=(w_{1},\dots,w_{M})^{\top}$ hold the fraction of that wealth placed in each asset at time $0$. For a __fully invested, long-only__ portfolio the weights are non-negative and sum to one, $w_{i}\geq0$ and $\sum_{i\in\mathcal{P}}w_{i}=1$, so the feasible set of weight vectors is the $(M-1)$-dimensional __simplex__ $\Delta^{M-1}$: a line segment for two assets, a triangle for three, and so on. Buying $n_{i}=w_{i}W_{0}/S_{i}(0)$ (fractional) shares of asset $i$ and holding them without trading, the __buy-and-hold__ portfolio wealth at time $t$ is:
$$
\boxed{
\begin{align*}
W_{t} &= \sum_{i\in\mathcal{P}}n_{i}\,S_{i}(t) = W_{0}\sum_{i\in\mathcal{P}}w_{i}\,\frac{S_{i}(t)}{S_{i}(0)}\\
\end{align*}}
$$
Here $w_{i}$ are initial dollar fractions and fractional shares are allowed. The initial weights are only initial: as relative prices move, the fraction of wealth in each asset, $w_{i}(t)=n_{i}S_{i}(t)/W_{t}$, drifts away from $\mathbf{w}$. A __constant-weight__ portfolio that trades back to $\mathbf{w}$ is a different, dynamically rebalanced strategy with its own turnover and cost assumptions.

How should we choose $\mathbf{w}$? Next lecture (L5b) we optimize it. Today we only want a way to __explore__ the simplex, that is, to draw many valid weight vectors and see what portfolios they produce, and for that we use the Dirichlet distribution.

> __Dirichlet portfolio weights__
>
> Let $\boldsymbol{\alpha}=(\alpha_{1},\ldots,\alpha_{M})$ with every __concentration parameter__ $\alpha_{i}>0$, and let $\alpha_{0}=\sum_{i\in\mathcal{P}}\alpha_{i}$. A random vector $\mathbf{W}\sim\text{Dirichlet}(\boldsymbol{\alpha})$ lies in the interior of the simplex $\Delta^{M-1}$ (its components are strictly positive and sum to one; exact zero weights never occur) and has component mean, variance, and covariance:
> $$
\begin{align*}
\mathbb{E}\left[W_{i}\right] = \frac{\alpha_{i}}{\alpha_{0}},\qquad
\text{Var}\left(W_{i}\right) = \frac{\alpha_{i}\left(\alpha_{0}-\alpha_{i}\right)}{\alpha_{0}^{2}\left(\alpha_{0}+1\right)},\qquad
\text{Cov}\left(W_{i},W_{j}\right) = -\frac{\alpha_{i}\alpha_{j}}{\alpha_{0}^{2}\left(\alpha_{0}+1\right)}\quad(i\neq j)
\end{align*}
$$
> The covariances are negative because the weights compete for the same unit of wealth. When every $\alpha_{i}=1$ the density is constant with respect to volume on the simplex (no region of the simplex is favored over another of the same size); the components are not uniform separately (each marginal is $\text{Beta}(1,M-1)$).

The concentration vector is the dial. With symmetric concentrations $\alpha_{i}=\alpha$, values below one favor allocations near the boundary (portfolios concentrated in a few assets) relative to the uniform case, while large values cluster the draws near equal weights $1/M$; unequal $\alpha_{i}$ tilt the draws toward the assets with larger concentration. None of these draws is __optimal__ in any sense; a Dirichlet draw is a candidate, and it becomes good or bad only when we evaluate an objective for it. The simplest objective uses the one-step growth rate of a portfolio held at weights $\mathbf{w}$: to first order it is the weighted growth rate $g_{p}=\mathbf{w}^{\top}\mathbf{g}$, with mean $\mathbf{w}^{\top}\bar{\mathbf{g}}$ and variance $\mathbf{w}^{\top}\text{Cov}(\mathbf{g})\mathbf{w}$, estimated by $\mathbf{w}^{\top}\mathbf{g}^{\prime}$ and $\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g}\mathbf{w}$. (This is a one-period, fixed-weight statistic; the log growth of buy-and-hold wealth over many periods is not simply the weighted sum of the assets' log growth.) Sampling explores; L5b optimizes, and L5b's portfolio example simulates buy-and-hold wealth with correlated MAGBM paths for a chosen $\mathbf{w}$. Let's see what the dial does.

> __Example__
>
> [▶ Sample portfolio weights with the Dirichlet distribution](CHEME-5660-L5a-Example-Dirichlet-PortfolioWeights-Fall-2026.ipynb). Draw random long-only weight vectors for a small portfolio, see how the concentration parameter shapes them, and compute the growth-rate mean and variance of the resulting portfolios.

___


## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L5b; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ Sampling error and shrinkage in covariance estimation](advanced/covariance-estimation/CHEME-5660-L5a-Advanced-CovarianceEstimation-Fall-2026.ipynb). Measure how noisy a large sample covariance is (its eigenvalue spectrum against the Marchenko-Pastur law), shrink it toward a structured target, and see what that does to a minimum-variance portfolio.
* [▶ Rolling correlations](advanced/rolling-correlation/CHEME-5660-L5a-Advanced-RollingCorrelation-Fall-2026.ipynb). Track pairwise correlations through 2014 to 2024 with rolling and exponentially weighted windows, watch them spike in stressed markets, and see what a constant covariance rate misses.
___


## Summary
In this lecture, we extended geometric Brownian motion from one asset to many correlated assets, estimated the covariance that couples them from historical growth rates, and introduced Dirichlet sampling as a way to explore portfolio weights on the long-only simplex.

> __Key Takeaways:__
>
> * **Correlated assets need a shared shock and a matrix square root:** Multiple asset GBM drives every asset with the same vector of independent shocks, mixed by a loading matrix whose square is the covariance rate; each asset keeps its own half-variance correction, and the exact one-step transition redraws the shock vector at every step.
>
> * **The covariance is estimated from centered growth rates and scaled by the time step:** The sample covariance of the growth-rate data matrix is symmetric and positive semidefinite by construction; multiplying it by the time step gives the GBM covariance rate the model needs, and its diagonal returns the volatilities of L4b, while sampling error and singularity are checks the estimate must pass before it is used.
>
> * **Portfolio weights live on the simplex, and Dirichlet draws explore it:** Buy-and-hold wealth is a weighted sum of price ratios, the concentration vector controls whether random weights sit near the corners or near equal weights, and sampling weights is exploration rather than optimization.

Next time, we turn the estimated means and covariance into the minimum-variance portfolio and the efficient frontier.
___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___